In [0]:
dbutils.widgets.text(name="github_token",defaultValue="", label="GitHub Token")


In [0]:
import requests

github_token = dbutils.widgets.get("github_token")
headers = {"Authorization": f"token {github_token}"}
response = requests.get("https://api.github.com/user/repos", headers=headers)
repos = response.json()

import pandas as pd
df = pd.DataFrame([{"name": repo["name"], "url": repo["html_url"]} for repo in repos])
spark_df = spark.createDataFrame(df)
display(spark_df)

In [0]:
%sh pip install --upgrade databricks-sdk

In [0]:
%sh pip install --upgrade databricks-sql-connector

In [0]:
dbutils.widgets.text(name='db_token',defaultValue='')
dbutils.widgets.text(name='url',defaultValue='')

In [0]:
import os

os.environ["DB_GITHUB_TOKEN"] = dbutils.widgets.get("github_token")
os.environ["DATABRICKS_TOKEN"] = dbutils.widgets.get("db_token")
os.environ["DATABRICKS_HOST"] = dbutils.widgets.get("url")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient(
    host=os.environ["DATABRICKS_HOST"],
    token=os.environ["DATABRICKS_TOKEN"]
)

In [0]:
scope_name = "mysecrets_cli"

try:
  w.secrets.create_scope(scope_name)
  print(f"Created secret scope {scope_name} successfully!")
except Exception as e:
  print(f"Error in creating Secret scope {scope_name}. Error: {e}")

In [0]:
try:
  scopes=w.secrets.list_scopes()
  for scope in scopes:
      print(scope.name)
except Exception as e:
  print(f"Error in listing Secret scopes. Error: {e}")

In [0]:
scope_name="mysecrets_cli"
secrets_dict = {
    "github_token":os.environ["DB_GITHUB_TOKEN"]
}

try:
  for key,value in secrets_dict.items():
    w.secrets.put_secret(scope=scope_name,key=key,string_value=value)
    print(f"Added secret {key} to scope {scope_name} successfully!")
except Exception as e:
  print(f"Error in adding secrets to scope {scope_name}. Error: {e}")

In [0]:
scope_name="mysecrets_cli"
try:
  secrets=w.secrets.list_secrets(scope_name)
  print(f"Secrets in scope {scope_name}:")
  for secret in secrets:
      print(f" -{secret.key}")
except Exception as e:
  print(f"Error in listing secrets in scope {scope_name}. Error: {e}")

In [0]:
import requests

github_token = dbutils.secrets.get(scope="mysecrets_cli", key="github_token")
headers = {"Authorization": f"token {github_token}"}
response = requests.get("https://api.github.com/user/repos", headers=headers)
repos = response.json()

import pandas as pd
df = pd.DataFrame([{"name": repo["name"], "url": repo["html_url"]} for repo in repos])
spark_df = spark.createDataFrame(df)
display(spark_df)

In [0]:
%pip install --upgrade PyGithub

In [0]:
import os
from github import Github
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

GITHUB_TOKEN = dbutils.secrets.get(scope="mysecrets_cli", key="github_token")
if not GITHUB_TOKEN:
  raise Exception("Please set your GITHUB_TOKEN in the notebook's widget")
try:
    g = Github(GITHUB_TOKEN)
    user = g.get_user()
    repos = list(user.get_repos())
    repo_count = len(repos)
    print(f"Found {repo_count} repos")
except Exception as e:
    print(f"Error: {e}")

if repo_count == 0:
  raise Exception("No repos found. Please create a repo in your GitHub account and try again")
else:
  data = [(repo.name,repo.git_url,repo.created_at,repo.open_issues_count,repo.visibility,repo.watchers_count) for repo in repos]

  df = spark.createDataFrame(data, schema=["name", "url", "created_at", "open_issues_count", "visibility", "watchers_count"])
  display(df)


In [0]:
dbutils.secrets.get(scope="mysecrets_cli", key="github_token")

In [0]:
from databricks.sdk.service.workspace import AclPermission

scope_name = "mysecrets_cli"

#Define the user/group adn their permission level
permissions = [
  {"principal":"users","permission":AclPermission.READ},
  {"principal":"admins","permission":AclPermission.MANAGE}
]
#Grant access to secrets
try:
  for permission in permissions:
    w.secrets.put_acl(scope=scope_name, principal=permission["principal"], permission=permission["permission"])
    print(f"Added {permission['principal']} with {permission['permission']} permission to scope {scope_name}")
except Exception as e:
  print(f"Error in adding permissions to scope {scope_name}. Error: {e}")

In [0]:
scope_name = "mysecrets_cli"
try:
  scopes=w.secrets.list_acls(scope_name)
  print(f"ACLs in scope {scope_name}:")
  for scope in scopes:
      print(f" -{scope.principal} with {scope.permission} permission")
except Exception as e:
  print(f"Error in listing ACLs in scope {scope_name}. Error: {e}")

In [0]:
scope_name = "mysecrets_cli"

principals = ["users","admins"]
for principal in principals:
  try:
    w.secrets.delete_acl(scope=scope_name, principal=principal)
    print(f"Deleted {principal} from scope {scope_name}")
  except Exception as e:
    print(f"Error in deleting {principal} from scope {scope_name}. Error: {e}")


In [0]:
scope_name = "mysecrets_cli"
# delete scope above
try:
  w.secrets.delete_scope(scope_name)
  print(f"Deleted scope {scope_name}")
except Exception as e:
  print(f"Error in deleting scope {scope_name}. Error: {e}")

In [0]:
dbutils.widgets.help()

In [0]:
dbutils.widgets.removeAll()